In [1]:
import datasets
import torchvision.transforms.v2 as transforms
from torch.utils.data import DataLoader 


# SET PYTHONPATH to the parent directory 
import sys
import os
sys.path.append(os.path.abspath(".."))


/Users/karella/.conda/envs/hippy2d/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

ds = datasets.load_dataset("timm/resisc45", 
                           cache_dir='/Users/karella/Projects/rotation-invariant-neural-networks/hippy2d/data',
                           split='train')

t = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])
ds = ds.with_transform(t)
dl = DataLoader(ds, batch_size=32, shuffle=True)
batch = next(iter(dl))
batch['image'].shape
batch['label'].shape

/Users/karella/.conda/envs/hippy2d/lib/python3.11/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


torch.Size([32])

In [7]:
import numpy as np 
from sklearn.model_selection import train_test_split 
ds = datasets.load_dataset("dpdl-benchmark/colorectal_histology", split='train') 
labels = np.array([example['label'] for example in ds])
np.unique(labels)

array([0, 1, 2, 3, 4, 5, 6, 7])

In [4]:
train_idx, test_valid_idx = train_test_split(np.arange(len(labels)),
                                             test_size=0.2, 
                                             random_state=42,
                                             stratify=labels)
valid_idx, test_idx = train_test_split(test_valid_idx,
                                       test_size=0.5,
                                       random_state=42,
                                       stratify=labels[test_valid_idx])


In [5]:
import torch
from hippy2d.datasets import collate_tuple
import torchvision.transforms.v2 as transforms

transform = transforms.Compose([
    transforms.ToImage(),
    transforms.ToDtype(torch.float32)
])

dl = DataLoader(ds.with_transform(transform),
                batch_size=32,
                sampler=torch.utils.data.SubsetRandomSampler(valid_idx),
                collate_fn=collate_tuple)
counter = {}
for batch in dl:
    # Count the labels
    _, labels = batch
    unique, counts = np.unique(labels.numpy(), return_counts=True)
    for u, c in zip(unique, counts):
        counter[u] = counter.get(u, 0) + c
print(counter)


{0: 63, 1: 63, 2: 62, 3: 62, 4: 62, 5: 63, 6: 62, 7: 63}


In [2]:
from hippy2d.datasets import ColorectalHistology

dataset = ColorectalHistology(data_dir="/Users/karella/Projects/rotation-invariant-neural-networks/hippy2d/data")
dataset.prepare_data()
dataset.setup("fds")


In [3]:
train_ld = dataset.train_dataloader()
test_ld = dataset.test_dataloader()
valid_ld = dataset.val_dataloader()

In [4]:

counter = {}
for batch in valid_ld:
    # Count the labels
    _, labels = batch
    unique, counts = np.unique(labels.numpy(), return_counts=True)
    for u, c in zip(unique, counts):
        counter[u] = counter.get(u, 0) + c
print(counter)

NameError: name 'np' is not defined

In [20]:
# Get all training data
images = [batch[0].numpy() for batch in train_ld]

In [39]:
mean_histology = np.mean(np.concatenate(images, axis=0), axis=(0, -2, -1))
mean_variance = np.var(np.concatenate(images, axis=0), axis=(0, -2, -1))
mean_histology  

array([0.6498576 , 0.47195938, 0.5839992 ], dtype=float32)

In [ ]:
doe